# Aim 2 Sleep-Filter Recovery — 7-Phase Batch Re-Fit

**Context:** A filter-inversion bug in `wake_mask` (since fixed) caused several
fit scripts to silently train on wake epochs instead of sleep-only. This
notebook re-fits all affected models under the corrected sleep-only protocol.

**Recovery design:** `Thesis Vault/Project/2026-05-02 - sleep-filter recovery
design.md` (vault commit 9cc7b61). Plan: same dir, `... recovery plan.md`
(vault commit f361923). Code fixes: GitHub commit `1d136f4` or later.

**Phases (each saves to Drive on completion):**
1. 8.5-tax × 3 configs (full / physio_only / aasm_only) — anchor numbers
2. Lit-2019 sleep-only — honest published-comparable floor
3+4. Phase 1 batch — audit-v6 + physio_only cumulative ablations (8 fits)
5. v6 past-only 2×2 (4 fits)
6. v8 model-family ladder under FIXED HP, sleep-only (lightgbm/xgboost/catboost/rf/logreg/ensemble)
7. Phase 1 Exp 6 TabNet × 3 configs (GPU runtime needed)

**Runtime expectations:**
- Cells 1-10: Colab Pro+ High-RAM CPU (~4 h total)
- Cell 11: Switch to GPU (T4 or A100) (~1 h)
- Total wall clock: ~5 h

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
DRIVE_ROOT = '/content/drive/MyDrive/Thesis'
assert os.path.exists(DRIVE_ROOT), f'Expected {DRIVE_ROOT} to exist'
# Ensure logs dir exists on Drive
os.makedirs(f'{DRIVE_ROOT}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
print(f'Drive root OK: {DRIVE_ROOT}')

In [ ]:
# 2. Pull repo from GitHub at commit 1d136f4 or later (code-fixes commit)
from google.colab import userdata
PAT = userdata.get('GITHUB_PAT_THESIS')
REPO = 'jameswudo1019hack/thesis-bme-pipeline'
import subprocess, pathlib

if pathlib.Path('/content/Code').exists():
    subprocess.run(['git', '-C', '/content/Code', 'pull', '--ff-only'], check=True)
else:
    url = f'https://{PAT}@github.com/{REPO}.git'
    subprocess.run(['git', 'clone', '--depth=50', url, '/content/Code'], check=True)

# Verify we are at 1d136f4 or a descendant
result = subprocess.run(
    ['git', '-C', '/content/Code', 'merge-base', '--is-ancestor', '1d136f4', 'HEAD'],
    capture_output=True
)
assert result.returncode == 0, 'HEAD is not at or after 1d136f4 — pull or reset failed'
print('Repo state:')
subprocess.run(['git', '-C', '/content/Code', 'log', '-1', '--oneline'], check=True)
print('sleep-filter fix commit present ✓')

In [ ]:
# 3. Pull features tar from Drive and extract to /content/Code/features/
import shutil, subprocess, time, os

TAR = f'{DRIVE_ROOT}/data/features-phase1batch-v1.tar'
DST = '/content/Code/features'
assert os.path.exists(TAR), f'Tar not found at {TAR} — upload it first'

if os.path.exists(DST):
    shutil.rmtree(DST)
os.makedirs(DST, exist_ok=True)

t0 = time.time()
subprocess.run(['tar', '-xf', TAR, '-C', '/content/Code'], check=True)
print(f'Extract done in {time.time()-t0:.0f}s')
n_parquets = len([f for f in os.listdir(DST) if f.endswith('.parquet') and not f.startswith('._')])
print(f'  parquets found: {n_parquets}')
assert n_parquets > 0, 'No parquet files found after extraction'

In [ ]:
# 4. Install deps + thesis_pipeline (editable) + pytorch-tabnet (needed for cell 11)
!pip install -q lightgbm xgboost catboost pyarrow neurokit2 mne lxml click scikit-learn pytorch-tabnet 2>&1 | tail -5
%cd /content/Code
!pip install -q -e . 2>&1 | tail -3
print('All deps installed ✓')

In [ ]:
# 5. Sanity check: cohort version distribution + subject metadata sanity
%cd /content/Code
!python3 scripts/check_features_versions.py 2>&1 | tail -5

import pandas as pd
sm = pd.read_parquet('features/subject_metadata.parquet')
print(f'\nsubject_metadata.parquet sanity:')
print(f'  rows: {len(sm)}')
print(f'  TST median: {sm["tst_min"].median():.0f} min  (expect 320-400 for sleep-only; <200 = wake-only bug)')
print(f'  sleep_efficiency median: {sm["sleep_efficiency"].median():.3f}  (expect 0.7-0.85)')
assert sm["tst_min"].median() > 250, 'TST too low — sleep filter may still be inverted'
print('\u2713 Cohort + metadata look correctly sleep-filtered')

In [ ]:
# Phase 1: 8.5-tax × 3 configs (full / physio_only / aasm_only) — anchor numbers
%cd /content/Code
!python3 scripts/fit_aim2_v85_taxonomy_ablation.py 2>&1 | tee /content/phase1_v85tax.log | tail -100

# Checkpoint sync to Drive immediately
import shutil, os
SRC = '/content/Code/models/aim2_v85_taxonomy'
DST = '/content/drive/MyDrive/Thesis/results/aim2_v85_taxonomy'
if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'Phase 1 (v85 taxonomy) results → synced to Drive: {DST}')
else:
    print('WARNING: SRC not found — script may have failed')
if os.path.exists('/content/phase1_v85tax.log'):
    shutil.copy('/content/phase1_v85tax.log', '/content/drive/MyDrive/Thesis/logs/')
print('Phase 1 checkpoint done')

In [ ]:
# Phase 2: Lit-2019 sleep-only — honest published-comparable floor
%cd /content/Code
!python3 scripts/fit_aim2_lit_replication.py 2>&1 | tee /content/phase2_lit2019.log | tail -100

# Checkpoint sync to Drive immediately
import shutil, os
SRC = '/content/Code/models/aim2_lit_replication'
DST = '/content/drive/MyDrive/Thesis/results/aim2_lit_replication'
if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'Phase 2 (Lit-2019) results → synced to Drive: {DST}')
else:
    print('WARNING: SRC not found — script may have failed')
if os.path.exists('/content/phase2_lit2019.log'):
    shutil.copy('/content/phase2_lit2019.log', '/content/drive/MyDrive/Thesis/logs/')
print('Phase 2 checkpoint done')

In [ ]:
# Phase 3+4: Phase 1 batch — audit-v6 cumulative ablations, two feature modes
# Phase 3: full feature mode (audit-v6)
%cd /content/Code
!python3 scripts/fit_aim2_phase1_batch.py --feature-mode full 2>&1 | tee /content/phase3_p1batch_full.log | tail -100

# Phase 4: physio_only feature mode
!python3 scripts/fit_aim2_phase1_batch.py --feature-mode physio_only 2>&1 | tee /content/phase4_p1batch_physio.log | tail -100

# Checkpoint sync — both modes to Drive
import shutil, os, time

for mode, log_path in [
    ('full', '/content/phase3_p1batch_full.log'),
    ('physio_only', '/content/phase4_p1batch_physio.log'),
]:
    sub = '' if mode == 'full' else f'/{mode}'
    SRC = f'/content/Code/models/aim2_phase1_batch{sub}'
    DST = f'/content/drive/MyDrive/Thesis/results/aim2_phase1_batch{sub}'
    if os.path.exists(SRC):
        if os.path.exists(DST):
            shutil.rmtree(DST)
        shutil.copytree(SRC, DST)
        print(f'Phase 3+4 ({mode}) results → synced to Drive: {DST}')
    else:
        print(f'WARNING: SRC not found for mode={mode}')
    if os.path.exists(log_path):
        shutil.copy(log_path, f'/content/drive/MyDrive/Thesis/logs/')
print('Phase 3+4 checkpoint done')

In [ ]:
# Phase 5: v6 past-only 2x2 — 4 variants
# (1) default (past-only, no future, no v6-schema match)
%cd /content/Code
!python3 scripts/fit_aim2_v6_past_only.py 2>&1 | tee /content/phase5_v6_default.log | tail -80

# (2) include future context
!python3 scripts/fit_aim2_v6_past_only.py --include-future-context 2>&1 | tee /content/phase5_v6_future.log | tail -80

# (3) match v6 schema
!python3 scripts/fit_aim2_v6_past_only.py --match-v6-schema 2>&1 | tee /content/phase5_v6_schema.log | tail -80

# (4) match v6 schema + include future context
!python3 scripts/fit_aim2_v6_past_only.py --match-v6-schema --include-future-context 2>&1 | tee /content/phase5_v6_schema_future.log | tail -80

# Checkpoint sync — all 4 variants to Drive
import shutil, os

SRC = '/content/Code/models/aim2_v6_past_only'
DST = '/content/drive/MyDrive/Thesis/results/aim2_v6_past_only'
if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'Phase 5 (v6 past-only 2x2) results → synced to Drive: {DST}')
else:
    print('WARNING: SRC aim2_v6_past_only not found — script may have failed')

for log_name in ['phase5_v6_default.log', 'phase5_v6_future.log', 'phase5_v6_schema.log', 'phase5_v6_schema_future.log']:
    p = f'/content/{log_name}'
    if os.path.exists(p):
        shutil.copy(p, '/content/drive/MyDrive/Thesis/logs/')
print('Phase 5 checkpoint done')

In [ ]:
# Phase 6: v8 model-family ladder (FIXED HP, sleep-only) — lightgbm/xgboost/catboost/rf/logreg/ensemble
%cd /content/Code
!python3 scripts/fit_aim2_v8_fixedhp_sleep.py --model all 2>&1 | tee /content/phase6_v8_fixedhp.log | tail -100

# Checkpoint sync to Drive immediately
import shutil, os
SRC = '/content/Code/models/aim2_v8_fixedhp_sleep'
DST = '/content/drive/MyDrive/Thesis/results/aim2_v8_fixedhp_sleep'
if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'Phase 6 (v8 fixed-HP ladder) results → synced to Drive: {DST}')
else:
    print('WARNING: SRC aim2_v8_fixedhp_sleep not found — script may have failed')
if os.path.exists('/content/phase6_v8_fixedhp.log'):
    shutil.copy('/content/phase6_v8_fixedhp.log', '/content/drive/MyDrive/Thesis/logs/')
print('Phase 6 checkpoint done')

In [ ]:
# Phase 7: Phase 1 Exp 6 TabNet × 3 configs
# NOTE: This cell requires a GPU runtime (T4 or A100).
# Before running: Runtime → Change runtime type → GPU (T4 or A100), then rerun cells 1-4.

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Switch to GPU runtime (Runtime -> Change runtime type -> GPU).')
print(f'GPU detected: {result.stdout.strip()}')

%cd /content/Code
!python3 scripts/fit_aim2_phase1_exp6_tabnet.py 2>&1 | tee /content/phase7_tabnet.log | tail -100

# Checkpoint sync to Drive immediately
import shutil, os
SRC = '/content/Code/models/aim2_phase1_exp6_tabnet'
DST = '/content/drive/MyDrive/Thesis/results/aim2_phase1_exp6_tabnet'
if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'Phase 7 (TabNet) results → synced to Drive: {DST}')
else:
    print('WARNING: SRC aim2_phase1_exp6_tabnet not found — script may have failed')
if os.path.exists('/content/phase7_tabnet.log'):
    shutil.copy('/content/phase7_tabnet.log', '/content/drive/MyDrive/Thesis/logs/')
print('Phase 7 checkpoint done')

In [ ]:
# Final sync: copy ALL local results to Drive (catches anything not yet synced)
import shutil, os, glob

LOCAL_MODELS = '/content/Code/models'
DRIVE_RESULTS = '/content/drive/MyDrive/Thesis/results'

synced = 0
for src_dir in sorted(glob.glob(f'{LOCAL_MODELS}/aim2_*')):
    run_name = os.path.basename(src_dir)
    dst_dir = os.path.join(DRIVE_RESULTS, run_name)
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    n_files = sum(len(fs) for _, _, fs in os.walk(dst_dir))
    print(f'  {run_name:50s} → {n_files} files')
    synced += 1

print(f'\nFinal sync complete: {synced} run directories synced to Drive')

In [ ]:
# Compact summary across all 7 phases
import json, glob, os
from pathlib import Path
DRIVE_RESULTS = '/content/drive/MyDrive/Thesis/results'
print(f"{'run':55s}  {'AUC':>6s}  {'filter':>14s}  {'n_feat':>6s}")
print('-' * 90)
for path in sorted(glob.glob(f'{DRIVE_RESULTS}/aim2_*/**/metrics.json', recursive=True)):
    try:
        m = json.load(open(path))
        run = path.replace(DRIVE_RESULTS + '/', '').replace('/metrics.json', '')
        auc = m.get('test_auc_roc') or m.get('auc_roc', 'n/a')
        filt = m.get('filter_used', '?')
        n_feat = m.get('n_features', '?')
        print(f"{run:55s}  {auc:>6.4f}  {filt:>14s}  {n_feat:>6}")
    except Exception:
        pass